# Выполнение ЛР №5: Классификация и регрессия

## Подключение библиотек

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт модулей sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import ElasticNet

# Дополнительные импорты для обработки данных
import warnings
warnings.filterwarnings('ignore')

## Настройка библиотек

In [ ]:
# Настройка numpy для воспроизводимости результатов
np.random.seed(42)

## Задание 1: Классификация kNN на датасете flame

### Формулировка

Выполнить классификацию методом k ближайших соседей на датасете flame:
1. Загрузить и разобрать данные из файла flame.txt
1. Разделить данные flame на обучающий и тестовый наборы
1. Оценить точность для разных значений k от 2 до 20 с использованием кросс-валидации
1. Построить график зависимости точности от k

### Решение

#### 1.1 Загрузка и парсинг данных flame

In [ ]:
# Загрузка данных из файла flame.txt
flame_data_path = '../ЛР (4)/Вариант 4/flame.txt'

# Чтение данных с разделителем табуляция
flame_data = pd.read_csv(flame_data_path, sep='\t', header=None, names=['x1', 'x2', 'class'])

print("Данные flame успешно загружены!")
print(f"Размер датасета: {flame_data.shape}")
print("\nПервые 10 строк:")
display(flame_data.head(10))

# Разделение данных на признаки (X) и метки классов (y)
X_flame = flame_data[['x1', 'x2']].values
y_flame = flame_data['class'].values

print(f"\nРазмер матрицы признаков X: {X_flame.shape}")
print(f"Размер вектора меток y: {y_flame.shape}")
print(f"Уникальные классы: {np.unique(y_flame)}")

#### 1.2 Разделение данных на обучающий и тестовый наборы

In [ ]:
# Разделение данных flame на обучающий и тестовый наборы
# Используем фиксированный random_state для воспроизводимости результатов
X_train, X_test, y_train, y_test = train_test_split(
    X_flame, y_flame, 
    test_size=0.3,  # 30% данных для тестирования
    random_state=42,  # Фиксированное значение для воспроизводимости
    stratify=y_flame  # Сохранение пропорций классов
)

print("Данные успешно разделены на обучающий и тестовый наборы!")
print(f"Размер исходного датасета: {X_flame.shape[0]} образцов")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов ({X_train.shape[0]/X_flame.shape[0]*100:.1f}%)")
print(f"Размер тестового набора: {X_test.shape[0]} образцов ({X_test.shape[0]/X_flame.shape[0]*100:.1f}%)")

print("\nРаспределение классов в обучающем наборе:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_label, count in zip(unique_train, counts_train):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_train)*100:.1f}%)")

print("\nРаспределение классов в тестовом наборе:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_label, count in zip(unique_test, counts_test):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_test)*100:.1f}%)")

# Проверка корректности разделения
total_samples = X_train.shape[0] + X_test.shape[0]
print(f"\nПроверка: {X_train.shape[0]} + {X_test.shape[0]} = {total_samples} (исходно: {X_flame.shape[0]})")
assert total_samples == X_flame.shape[0], "Ошибка: потеря данных при разделении!"

#### 1.3 Оценка точности kNN для разных значений k

In [ ]:

# Оценка точности kNN для разных значений k от 2 до 20
from pandas import DataFrame


k_range = range(2, 21)  # k от 2 до 20
k_scores = []

print("Оценка точности kNN для разных значений k:")

# Цикл оценки для каждого k
accuracy_for_arr_k = []
for k in k_range:
    # Создание модели kNN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Кросс-валидация с 5 фолдами
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    
    # Сохранение среднего значения точности
    mean_accuracy = cv_scores.mean()
    std_accuracy = cv_scores.std()
    k_scores.append(mean_accuracy)
    
    accuracy_for_arr_k.append({
        'k': k,
        'Точность (среднее)': mean_accuracy,
        'Стандартное отклонение': std_accuracy,
    })
display(DataFrame(accuracy_for_arr_k).style.hide(axis='index'))

print(f"\nВсего оценено k значений: {len(k_scores)}")
print(f"Лучшая точность: {max(k_scores):.4f} при k = {k_range[k_scores.index(max(k_scores))]}")
print(f"Худшая точность: {min(k_scores):.4f} при k = {k_range[k_scores.index(min(k_scores))]}")

#### 1.4 Построение графика точности vs k

In [ ]:
# Построение графика зависимости точности от k
plt.figure(figsize=(12, 8))

# Основной график
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8, label='Точность кросс-валидации')

# Выделение максимального значения
best_k = k_range[k_scores.index(max(k_scores))]
best_score = max(k_scores)
plt.plot(best_k, best_score, 'ro', markersize=12, label=f'Лучший результат (k={best_k})')

# Настройка графика
plt.xlabel('Количество соседей (k)', fontsize=14)
plt.ylabel('Точность классификации', fontsize=14)
plt.title('Зависимость точности kNN от количества соседей k\n(датасет flame)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Установка диапазона осей
plt.xlim(1.5, 20.5)
plt.ylim(min(k_scores) - 0.02, max(k_scores) + 0.02)

# Настройка тиков на оси x
plt.xticks(range(2, 21, 2))

plt.tight_layout()
plt.show()

# Вывод статистики
print(f"\nСтатистика по результатам:")
print(f"Оптимальное значение k: {best_k}")
print(f"Максимальная точность: {best_score:.4f}")
print(f"Средняя точность по всем k: {np.mean(k_scores):.4f}")
print(f"Стандартное отклонение: {np.std(k_scores):.4f}")

#### Выводы по заданию 1

1. **Загрузка данных**: Успешно загружен датасет flame с двумя признаками (x1, x2) и двумя классами (1, 2)
1. **Разделение данных**: Данные flame успешно разделены на обучающий (70%) и тестовый (30%) наборы с сохранением пропорций классов
1. **Оценка kNN**: Проведена оценка точности для k от 2 до 20 с использованием 5-фолдовой кросс-валидации
1. **Визуализация**: Построен график зависимости точности от k, который показывает оптимальное значение k
1. **Результат**: Определено оптимальное значение k для данного датасета

## Задание 2: Визуализация данных с разделением на классы

### Формулировка

Визуализировать обучающие и тестовые данные с метками классов:
1. Создать диаграмму рассеяния с цветовым кодированием по классам
2. Различить обучающие и тестовые данные визуально
3. Добавить легенду с метками классов

### Решение

#### 2.1 Создание диаграммы рассеяния с классами

In [ ]:
# Определение цветов для классов
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# Дополнительная детальная визуализация
plt.figure(figsize=(12, 8))

# Создание графика с разделением
for class_label in np.unique(y_flame):
    # Обучающие данные для каждого класса
    mask_train = y_train == class_label
    plt.scatter(X_train[mask_train, 0], X_train[mask_train, 1], 
               c=colors[class_label], alpha=0.7, s=70, 
               label=f'{class_names[class_label]} - Обучение ({np.sum(mask_train)} точек)', 
               marker='o', edgecolors='darkgray', linewidth=0.8)
    
    # Тестовые данные для каждого класса
    mask_test = y_test == class_label
    plt.scatter(X_test[mask_test, 0], X_test[mask_test, 1], 
               c=colors[class_label], alpha=1.0, s=100, 
               label=f'{class_names[class_label]} - Тест ({np.sum(mask_test)} точек)', 
               marker='^', edgecolors='black', linewidth=1.2)

plt.xlabel('Признак x1', fontsize=14)
plt.ylabel('Признак x2', fontsize=14)
plt.title('Детальная визуализация данных flame с разделением на классы и наборы', fontsize=16)
plt.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

#### Выводы по заданию 2

1. **Визуализация классов**: Создана диаграмма рассеяния с цветовым кодированием по классам (красный - класс 1, синий - класс 2)
1. **Различение наборов**: Обучающие данные показаны кругами, тестовые - треугольниками для четкого визуального различения

## Задание 3: Анализ результатов классификации

### Формулировка

Проанализировать результаты классификации kNN:
1. Выбрать оптимальное значение k на основе результатов кросс-валидации
1. Обучить финальную модель kNN с оптимальным k
1. Построить матрицу ошибок (confusion matrix)

### Решение

#### 3.1 Обучение оптимальной модели kNN

In [ ]:
# Выбор оптимального значения k на основе результатов кросс-валидации
optimal_k = k_range[k_scores.index(max(k_scores))]
optimal_accuracy = max(k_scores)

print("=== ВЫБОР ОПТИМАЛЬНОГО ЗНАЧЕНИЯ K ===")
print(f"Оптимальное значение k: {optimal_k}")
print(f"Максимальная точность кросс-валидации: {optimal_accuracy:.4f}")

# Обучение финальной модели kNN с оптимальным k
print("\n=== ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ ===")
final_knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn_model.fit(X_train, y_train)

print(f"Модель kNN успешно обучена с k = {optimal_k}")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Количество классов: {len(np.unique(y_train))}")

# Проверка обученной модели
print("\n=== ПАРАМЕТРЫ ОБУЧЕННОЙ МОДЕЛИ ===")
print(f"Алгоритм: {final_knn_model.algorithm}")
print(f"Метрика расстояния: {final_knn_model.metric}")
print(f"Количество соседей: {final_knn_model.n_neighbors}")
print(f"Веса: {final_knn_model.weights}")

# Получение предсказаний на тестовом наборе
y_pred = final_knn_model.predict(X_test)
y_pred_proba = final_knn_model.predict_proba(X_test)

print("\n=== ПРЕДСКАЗАНИЯ НА ТЕСТОВОМ НАБОРЕ ===")
print(f"Количество тестовых образцов: {len(y_test)}")
print(f"Количество предсказаний: {len(y_pred)}")
print(f"Уникальные предсказанные классы: {np.unique(y_pred)}")
print(f"Уникальные истинные классы: {np.unique(y_test)}")

# Предварительная оценка точности
test_accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность на тестовом наборе: {test_accuracy:.4f}")
print(f"Разница с кросс-валидацией: {abs(test_accuracy - optimal_accuracy):.4f}")

#### 3.2 Вычисление метрик производительности

In [ ]:
# Построение матрицы ошибок (confusion matrix)
cm = confusion_matrix(y_test, y_pred)

# Визуализация матрицы ошибок
plt.figure(figsize=(10, 8))

# Создание heatmap для матрицы ошибок
unique_classes = np.unique(y_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Предсказан\nКласс {i}' for i in unique_classes],
            yticklabels=[f'Истинный\nКласс {i}' for i in unique_classes],
            cbar_kws={'label': 'Количество образцов'})

plt.title(f'Матрица ошибок для kNN (k={optimal_k})', fontsize=16)
plt.xlabel('Предсказанный класс', fontsize=14)
plt.ylabel('Истинный класс', fontsize=14)

# Добавление процентов в ячейки
total_samples = np.sum(cm)
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        percentage = cm[i, j] / total_samples * 100
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

# Анализ матрицы ошибок
print("\n--- АНАЛИЗ МАТРИЦЫ ОШИБОК ---")
total_correct = np.trace(cm)  # Сумма диагональных элементов
total_samples = np.sum(cm)
total_errors = total_samples - total_correct

print(f"Всего образцов: {total_samples}")
print(f"Правильно классифицировано: {total_correct} ({total_correct/total_samples*100:.1f}%)")
print(f"Неправильно классифицировано: {total_errors} ({total_errors/total_samples*100:.1f}%)")

#### Выводы по заданию 3

1. **Оптимальная модель**: Выбрано оптимальное значение k на основе кросс-валидации и обучена финальная модель kNN
3. **Матрица ошибок**: Построена и проанализирована confusion matrix

## Задание 4: Анализ ошибок классификации

### Формулировка

Проанализировать и визуализировать ошибки классификации kNN:
1. Идентифицировать неправильно классифицированные точки
1. Выделить эти точки на графике
1. Показать границы принятия решений

### Решение

#### 4.1 Идентификация неправильно классифицированных точек

In [ ]:

# Идентификация неправильно классифицированных точек
print("=== ИДЕНТИФИКАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Найти индексы неправильно классифицированных образцов
misclassified_mask = y_test != y_pred
misclassified_indices = np.where(misclassified_mask)[0]
correctly_classified_indices = np.where(~misclassified_mask)[0]

print(f"Всего тестовых образцов: {len(y_test)}")
print(f"Правильно классифицировано: {len(correctly_classified_indices)} ({len(correctly_classified_indices)/len(y_test)*100:.1f}%)")
print(f"Неправильно классифицировано: {len(misclassified_indices)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)")

# Сохранение данных об ошибках для дальнейшего использования
misclassified_points = X_test[misclassified_indices]
misclassified_true_labels = y_test[misclassified_indices]
misclassified_pred_labels = y_pred[misclassified_indices]

print("Индексы неправильно классифицированных точек:")
print(misclassified_indices)


correctly_classified_points = X_test[correctly_classified_indices]
correctly_classified_labels = y_test[correctly_classified_indices]

print(f"\nДанные об ошибках сохранены для визуализации:")
print(f"- Неправильно классифицированные точки: {misclassified_points.shape}")
print(f"- Правильно классифицированные точки: {correctly_classified_points.shape}")

#### 4.2 Создание визуализации ошибок

In [ ]:
# Создание визуализации ошибок классификации
print("=== ВИЗУАЛИЗАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Создание фигуры
fig, axes = plt.subplots(1, 1, figsize=(10, 8))

# Определение цветов для классов и типов предсказаний
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# График 1: Границы принятия решений
ax1 = axes

# Создание сетки для визуализации границ решений
h = 0.02  # Шаг сетки
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Предсказания для всех точек сетки
mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = final_knn_model.predict(mesh_points)
Z = Z.reshape(xx.shape)

# Отображение границ решений
ax1.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
ax1.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)

# Отображение тестовых точек
for class_label in np.unique(y_test):
    mask = y_test == class_label
    ax1.scatter(X_test[mask, 0], X_test[mask, 1], 
               c=colors[class_label], alpha=0.8, s=60, 
               label=f'{class_names[class_label]}', 
               marker='o', edgecolors='black', linewidth=0.5)

# Выделение ошибок
if len(misclassified_indices) > 0:
    ax1.scatter(misclassified_points[:, 0], misclassified_points[:, 1], 
               c='yellow', alpha=1.0, s=150, 
               label=f'Ошибки ({len(misclassified_indices)})', 
               marker='X', edgecolors='black', linewidth=2)

ax1.set_xlabel('Признак x1')
ax1.set_ylabel('Признак x2')
ax1.set_title('Границы принятия решений kNN')
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Выводы по заданию 4

1. На тестовом наборе данных ошибок не выявлено.
1. Создана визуализация границы принятия решений kNN с выделением ошибок

## Задание 5: Предобработка данных

### Формулировка

* Выполнить предобработку  датасета  из  корневого каталога по аналогии с третьей лабораторной работой

### Решение

#### 5.1 Загрузка и изучение датасета

In [ ]:
# Загрузка данных из файла train.csv
train_data_path = 'house-prices-advanced/train.csv'

# Чтение данных
house_data = pd.read_csv(train_data_path)

print("=" * 60)
print("ЗАГРУЗКА И ИЗУЧЕНИЕ ДАТАСЕТА HOUSE PRICES")
print("=" * 60)

print(f"\nРазмер датасета: {house_data.shape}")
print(f"Количество строк: {house_data.shape[0]}")
print(f"Количество признаков: {house_data.shape[1]}")

print("\n" + "=" * 60)
print("ИНФОРМАЦИЯ О ДАТАСЕТЕ")
print("=" * 60)
house_data.info()

print("\n" + "=" * 60)
print("ПЕРВЫЕ 5 СТРОК ДАТАСЕТА")
print("=" * 60)
display(house_data.head())

print("\n" + "=" * 60)
print("СТАТИСТИКА ПО ЧИСЛЕННЫМ ПРИЗНАКАМ")
print("=" * 60)
display(house_data.describe())

# Разделение на численные и категориальные признаки
numerical_cols = house_data.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = house_data.select_dtypes(include=['object']).columns.tolist()

# Удаляем Id и SalePrice из численных (Id - идентификатор, SalePrice - целевая переменная)
numerical_cols.remove('Id')
numerical_cols.remove('SalePrice')

print("\n" + "=" * 60)
print("АНАЛИЗ ТИПОВ ПРИЗНАКОВ")
print("=" * 60)
print(f"\nЧисленные признаки ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nКатегориальные признаки ({len(categorical_cols)}):")
print(categorical_cols)

# Сохраняем целевую переменную отдельно
y_house = house_data['SalePrice'].copy()
print(f"\nЦелевая переменная SalePrice:")
print(f"  Минимум: {y_house.min():,.0f}")
print(f"  Максимум: {y_house.max():,.0f}")
print(f"  Среднее: {y_house.mean():,.0f}")
print(f"  Медиана: {y_house.median():,.0f}")


#### 5.2 Анализ пропущенных значений

In [ ]:
# Создаем копию данных для обработки
house_data_processed = house_data

print("=" * 60)
print("АНАЛИЗ ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
print("=" * 60)

# Подсчет пропущенных значений
missing_values = house_data_processed.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if len(missing_values) > 0:
    missing_percent = (missing_values / len(house_data_processed)) * 100
    missing_df = pd.DataFrame({
        'Количество пропусков': missing_values,
        'Процент пропусков': missing_percent.round(2)
    })
    
    print(f"\nНайдено {len(missing_values)} признаков с пропущенными значениями:\n")
    display(missing_df)
    
    # Визуализация пропущенных значений
    plt.figure(figsize=(12, 8))
    top_missing = missing_df.head(20)  # Топ-20 признаков с пропусками
    plt.barh(range(len(top_missing)), top_missing['Процент пропусков'].values)
    plt.yticks(range(len(top_missing)), top_missing.index)
    plt.xlabel('Процент пропущенных значений (%)', fontsize=12)
    plt.title('Топ-20 признаков с пропущенными значениями', fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Анализ типов пропусков
    print("\n" + "=" * 60)
    print("АНАЛИЗ ПРОПУСКОВ ПО ТИПАМ ПРИЗНАКОВ")
    print("=" * 60)
    
    missing_numerical = [col for col in missing_values.index if col in numerical_cols]
    missing_categorical = [col for col in missing_values.index if col in categorical_cols]
    
    print(f"\nЧисленные признаки с пропусками ({len(missing_numerical)}):")
    if missing_numerical:
        for col in missing_numerical:
            print(f"  {col}: {missing_values[col]} пропусков ({missing_percent[col]:.2f}%)")
    else:
        print("  Нет")
    
    print(f"\nКатегориальные признаки с пропусками ({len(missing_categorical)}):")
    if missing_categorical:
        for col in missing_categorical:
            print(f"  {col}: {missing_values[col]} пропусков ({missing_percent[col]:.2f}%)")
    else:
        print("  Нет")
    
else:
    print("\n✓ Пропущенных значений не найдено!")


#### 5.3 Обработка пропущенных значений и преобразование категориальных данных

In [ ]:
# Импорт модуля предобработки данных
from preprocessing import DataPreprocessor

# Список признаков, где "NA" означает "None" (согласно описанию)
# Это признаки, связанные с отсутствием объектов (нет подвала, нет гаража и т.д.)
na_means_none = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature'
]

# Определяем порядковые признаки (качества и условия)
# Эти признаки имеют естественный порядок и могут быть закодированы численно
ordinal_features = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'Utilities': ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    'LandContour': ['Low', 'Bnk', 'HLS', 'Lvl']
}

# Создаем один экземпляр препроцессора для обработки пропусков и преобразования категориальных данных
preprocessor = DataPreprocessor(verbose=True)
house_data_processed = preprocessor.fit_transform(
    house_data_processed,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    ordinal_features=ordinal_features,
    na_means_none=na_means_none,
    exclude_cols=['Id', 'SalePrice']
)

# Финальная проверка пропущенных значений
print("\n" + "=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
print("=" * 60)

final_missing = house_data_processed.isnull().sum()
final_missing = final_missing[final_missing > 0]

if len(final_missing) == 0:
    print("✓ Все пропущенные значения успешно обработаны!")
else:
    print(f"⚠️ Остались пропуски в следующих признаках ({len(final_missing)}):")
    for col, count in final_missing.items():
        print(f"  {col}: {count} пропусков")

# Финальная проверка преобразований
print("\n" + "=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПРЕОБРАЗОВАНИЙ")
print("=" * 60)

print(f"\nРазмер датасета после преобразований: {house_data_processed.shape}")

# Проверяем, что все категориальные признаки преобразованы
remaining_categorical = house_data_processed.select_dtypes(include=['object']).columns.tolist()
if 'Id' in remaining_categorical:
    remaining_categorical.remove('Id')

if len(remaining_categorical) == 0:
    print("\n✓ Все категориальные признаки успешно преобразованы!")
else:
    print(f"\n⚠️ Остались категориальные признаки ({len(remaining_categorical)}):")
    print(remaining_categorical)

# Удаляем Id, если он есть (не нужен для модели)
if 'Id' in house_data_processed.columns:
    house_data_processed = house_data_processed.drop(columns=['Id'])


### Выводы

1. **Загрузка данных**: Успешно загружен датасет House Prices с 1460 строками и 81 признаком. Целевая переменная SalePrice сохранена отдельно.

2. **Анализ пропущенных значений**: 
   - Проанализированы все пропущенные значения по каждому признаку
   - Учтена особенность датасета: "NA" в некоторых полях означает категорию "None" (отсутствие объекта), а не пропуск
   - Визуализированы топ-20 признаков с наибольшим количеством пропусков

3. **Обработка пропущенных значений**:
   - **Численные признаки**: заполнены медианой (более устойчива к выбросам)
   - **Категориальные признаки**: 
     - Для признаков, где "NA" означает отсутствие объекта (Alley, BsmtQual, GarageType и др.) → заполнено значением "None"
     - Для остальных категориальных признаков → заполнено модой (наиболее частым значением)
   - Все пропущенные значения успешно обработаны

4. **Преобразование категориальных данных**:
   - **Ординальное кодирование**: применено для порядковых признаков (качества, условия) с естественным порядком (Ex > Gd > TA > Fa > Po)
   - **One-Hot Encoding**: применено для номинальных категориальных признаков (MSZoning, Neighborhood, HouseStyle и др.)
   - Все категориальные признаки успешно преобразованы в численные

5. **Результат**: Данные полностью подготовлены для дальнейшего анализа - отбора признаков и построения регрессионной модели (Эластичная сеть).


## Задание 6

### Формулировка

* Выполнить отбор переменных для регрессии

### Решение

In [ ]:
print("=" * 60)
print("ОТБОР ПРИЗНАКОВ НА ОСНОВЕ КОЭФФИЦИЕНТА КОРРЕЛЯЦИИ")
print("=" * 60)

# 1. Подготовка данных для анализа корреляции
print("\n1. ПОДГОТОВКА ДАННЫХ ДЛЯ АНАЛИЗА КОРРЕЛЯЦИИ")
print("-" * 60)

# Создаем временный датасет с целевой переменной для расчета корреляций
# Проверяем, что SalePrice не входит в house_data_processed
if 'SalePrice' in house_data_processed.columns:
    house_data_for_corr = house_data_processed.copy()
else:
    # Добавляем SalePrice обратно для расчета корреляций
    house_data_for_corr = house_data_processed.copy()
    house_data_for_corr['SalePrice'] = y_house.values

print(f"Размер датасета для анализа корреляции: {house_data_for_corr.shape}")
print(f"Количество признаков (включая SalePrice): {house_data_for_corr.shape[1]}")
print(f"Количество строк: {house_data_for_corr.shape[0]}")

# Проверяем, что все признаки численные
non_numeric = house_data_for_corr.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric) > 0:
    print(f"\n⚠️ Обнаружены нечисленные признаки: {non_numeric}")
else:
    print("\n✓ Все признаки численные")

# 2. Расчет коэффициентов корреляции
print("\n" + "=" * 60)
print("2. РАСЧЕТ КОЭФФИЦИЕНТОВ КОРРЕЛЯЦИИ")
print("=" * 60)

# Вычисляем корреляционную матрицу
correlation_matrix = house_data_for_corr.corr()

# Извлекаем корреляции с целевой переменной SalePrice
correlations_with_target = correlation_matrix['SalePrice'].drop('SalePrice')

# Сортируем по абсолютному значению корреляции
correlations_sorted = correlations_with_target.abs().sort_values(ascending=False)

print(f"\nВсего признаков для анализа: {len(correlations_sorted)}")
print(f"Максимальная корреляция: {correlations_sorted.max():.4f}")
print(f"Минимальная корреляция: {correlations_sorted.min():.4f}")
print(f"Средняя абсолютная корреляция: {correlations_sorted.mean():.4f}")

# 3. Отбор признаков
# Применяем порог |correlation| > 0.3
correlation_threshold = 0.3

print("\n" + "=" * 60)
print(f"3. ОТБОР ПРИЗНАКОВ (порог: |correlation| > {correlation_threshold:2})")
print("=" * 60)

selected_features_mask = correlations_sorted > correlation_threshold
selected_features = correlations_sorted[selected_features_mask].index.tolist()

print(f"\nПорог отбора: |correlation| > {correlation_threshold}")
print(f"Отобрано признаков: {len(selected_features)} из {len(correlations_sorted)}")
print(f"Процент отобранных признаков: {len(selected_features)/len(correlations_sorted)*100:.1f}%")

# Создаем DataFrame с отобранными признаками и их корреляциями
# Сортируем по абсолютному значению корреляции (сначала получаем индексы, затем значения)
selected_features_sorted = correlations_with_target[selected_features].abs().sort_values(ascending=False).index
selected_correlations = correlations_with_target[selected_features_sorted]
selected_features_df = pd.DataFrame({
    'Признак': selected_correlations.index,
    'Корреляция': selected_correlations.values,
    '|Корреляция|': selected_correlations.abs().values
}).sort_values('|Корреляция|', ascending=False)

print("\nОтобранные признаки (топ-20):")
display(selected_features_df.head(20).style.hide(axis='index'))

# 4. Визуализация результатов
print("\n" + "=" * 60)
print("4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("=" * 60)

# График 1: Топ-20 признаков с наибольшей корреляцией
plt.figure(figsize=(12, 10))
top_20_features = selected_correlations.head(20)
colors_bar = ['red' if x < 0 else 'blue' for x in top_20_features.values]

plt.barh(range(len(top_20_features)), top_20_features.values, color=colors_bar, alpha=0.7)
plt.yticks(range(len(top_20_features)), top_20_features.index)
plt.xlabel('Коэффициент корреляции с SalePrice', fontsize=12)
plt.ylabel('Признаки', fontsize=12)
plt.title(f'Топ-20 признаков с наибольшей корреляцией с SalePrice\n(порог отбора: |corr| > {correlation_threshold})', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.axvline(x=correlation_threshold, color='green', linestyle='--', linewidth=1, label=f'Порог: {correlation_threshold}')
plt.axvline(x=-correlation_threshold, color='green', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3, axis='x')
plt.legend()
plt.tight_layout()
plt.show()

# График 2: Распределение корреляций
plt.figure(figsize=(12, 6))
plt.hist(correlations_with_target.values, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
plt.axvline(x=correlation_threshold, color='red', linestyle='--', linewidth=2, label=f'Порог: {correlation_threshold}')
plt.axvline(x=-correlation_threshold, color='red', linestyle='--', linewidth=2)
plt.xlabel('Коэффициент корреляции с SalePrice', fontsize=12)
plt.ylabel('Количество признаков', fontsize=12)
plt.title('Распределение коэффициентов корреляции всех признаков с SalePrice', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 5. Подготовка данных для модели
print("\n" + "=" * 60)
print("5. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ")
print("=" * 60)

# Создаем X_selected с отобранными признаками
X_selected = house_data_processed[selected_features].copy()

print(f"\nРазмерность данных до отбора: {house_data_processed.shape}")
print(f"Размерность данных после отбора: {X_selected.shape}")
print(f"Сокращение признаков: {house_data_processed.shape[1] - X_selected.shape[1]} признаков")
print(f"Процент сокращения: {(house_data_processed.shape[1] - X_selected.shape[1])/house_data_processed.shape[1]*100:.1f}%")

# Проверяем, что нет пропусков
if X_selected.isnull().sum().sum() == 0:
    print("\n✓ Пропущенных значений в отобранных признаках нет")
else:
    print("\n⚠️ Обнаружены пропущенные значения:")
    print(X_selected.isnull().sum()[X_selected.isnull().sum() > 0])

# Сохраняем список отобранных признаков для использования в задании 7
print(f"\n✓ Список отобранных признаков сохранен в переменной 'selected_features'")
print(f"✓ Данные с отобранными признаками сохранены в переменной 'X_selected'")
print(f"✓ Целевая переменная сохранена в переменной 'y_house'")

# 6. Документация результатов
print("\n" + "=" * 60)
print("6. ДОКУМЕНТАЦИЯ РЕЗУЛЬТАТОВ")
print("=" * 60)

print("\nПолный список отобранных признаков с корреляциями:")
print("=" * 60)
display(selected_features_df.style.hide(axis='index'))


### Выводы

1. **Расчет корреляций**: Вычислены коэффициенты корреляции Пирсона между всеми признаками и целевой переменной SalePrice

2. **Отбор признаков**: Применен порог |correlation| > 0.3 для отбора наиболее информативных признаков.

3. **Визуализация**: 
   - Построена столбчатая диаграмма топ-20 признаков с наибольшей корреляцией
   - Построена гистограмма распределения всех корреляций для понимания общего распределения

4. **Подготовка данных**: 
   - Создана матрица `X_selected` с отобранными признаками
   - Сохранен список `selected_features` для использования в следующем задании
   - Данные готовы для построения модели Эластичная сеть

5. **Результат**: Отобранные признаки будут использованы в задании 7 для построения регрессионной модели.


## Задание 7

### Формулировка

* Создать регрессионную модель используя метод "Эластичная сеть"

### Решение

In [ ]:
print("=" * 60)
print("ЗАДАНИЕ 7: СОЗДАНИЕ РЕГРЕССИОННОЙ МОДЕЛИ ELASTIC NET")
print("=" * 60)

# 1. Подготовка данных для обучения
print("\n1. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ")
print("-" * 60)

# Проверяем наличие данных из задания 6
print(f"Размер X_selected: {X_selected.shape}")
print(f"Размер y_house: {y_house.shape}")
print(f"Количество отобранных признаков: {len(selected_features)}")

# Разделяем данные на train и validation для оценки модели
# Используем 80% для обучения, 20% для валидации
X_train_elastic, X_val_elastic, y_train_elastic, y_val_elastic = train_test_split(
    X_selected, y_house,
    test_size=0.2,
    random_state=42
)

print(f"\nРазделение данных:")
print(f"  Обучающая выборка: {X_train_elastic.shape[0]} образцов ({X_train_elastic.shape[0]/len(X_selected)*100:.1f}%)")
print(f"  Валидационная выборка: {X_val_elastic.shape[0]} образцов ({X_val_elastic.shape[0]/len(X_selected)*100:.1f}%)")
print(f"  Количество признаков: {X_train_elastic.shape[1]}")

# 2. Создание и обучение модели Elastic Net
print("\n" + "=" * 60)
print("2. СОЗДАНИЕ И ОБУЧЕНИЕ МОДЕЛИ ELASTIC NET")
print("=" * 60)

# Параметры модели Elastic Net
alpha = 1.0  # Параметр регуляризации
l1_ratio = 0.5  # Соотношение L1/L2 (0.5 = равное сочетание Lasso и Ridge)

print(f"\nПараметры модели:")
print(f"  alpha (регуляризация): {alpha}")
print(f"  l1_ratio (L1/L2 соотношение): {l1_ratio}")
print(f"  (0 = Ridge, 1 = Lasso, 0.5 = Elastic Net)")

# Создание модели
elastic_net_model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=10000)

print("\nОбучение модели...")
# Обучение модели
elastic_net_model.fit(X_train_elastic, y_train_elastic)

print("✓ Модель успешно обучена!")

# 3. Параметры обученной модели
print("\n" + "=" * 60)
print("3. ПАРАМЕТРЫ ОБУЧЕННОЙ МОДЕЛИ")
print("=" * 60)

print(f"\nПараметры модели:")
print(f"  alpha: {elastic_net_model.alpha}")
print(f"  l1_ratio: {elastic_net_model.l1_ratio}")
print(f"  max_iter: {elastic_net_model.max_iter}")
print(f"  n_iter_: {elastic_net_model.n_iter_}")

# Количество ненулевых коэффициентов (важные признаки)
non_zero_coef = np.sum(elastic_net_model.coef_ != 0)
print(f"\nСтатистика коэффициентов:")
print(f"  Всего признаков: {len(elastic_net_model.coef_)}")
print(f"  Ненулевых коэффициентов: {non_zero_coef} ({non_zero_coef/len(elastic_net_model.coef_)*100:.1f}%)")
print(f"  Нулевых коэффициентов: {len(elastic_net_model.coef_) - non_zero_coef}")

# Топ-10 признаков с наибольшими по модулю коэффициентами
coef_abs = np.abs(elastic_net_model.coef_)
top_features_idx = np.argsort(coef_abs)[::-1][:10]
print(f"\nТоп-10 признаков с наибольшими коэффициентами:")
for i, idx in enumerate(top_features_idx, 1):
    feature_name = selected_features[idx]
    coef_value = elastic_net_model.coef_[idx]
    print(f"  {i}. {feature_name}: {coef_value:.4f}")

print(f"\n✓ Модель Elastic Net готова для использования!")


### Выводы

1. **Подготовка данных**: Данные из задания 6 (X_selected и y_house) разделены на обучающую (80%) и валидационную (20%) выборки для оценки модели

2. **Создание модели Elastic Net**: 
   - Создана модель с параметрами alpha=1.0 и l1_ratio=0.5 (равное сочетание L1 и L2 регуляризации)
   - Модель успешно обучена на обучающей выборке

3. **Параметры модели**: 
   - Модель использует комбинацию Lasso (L1) и Ridge (L2) регуляризации
   - Elastic Net помогает справиться с мультиколлинеарностью и отбором признаков
   - Часть коэффициентов обнулена благодаря L1-регуляризации

4. **Результат**: Модель готова для предсказаний и оценки качества на тестовых данных


## Задание 8

### Формулировка

* Для модели из задания 7 вывести среднеквадратическую ошибку (RMSE)

### Решение

In [ ]:
print("=" * 60)
print("ЗАДАНИЕ 8: ВЫЧИСЛЕНИЕ RMSE И ВИЗУАЛИЗАЦИЯ")
print("=" * 60)

# 1. Предобработка тестовых данных
print("\n1. ПРЕДОБРАБОТКА ТЕСТОВЫХ ДАННЫХ")
print("-" * 60)

# Загрузка тестовых данных
test_data_path = 'house-prices-advanced/test.csv'
test_data = pd.read_csv(test_data_path)

print(f"Загружено тестовых данных: {test_data.shape}")
print(f"Количество строк: {test_data.shape[0]}")
print(f"Количество признаков: {test_data.shape[1]}")

# Сохраняем Id для сопоставления
test_ids = test_data['Id'].copy()

# Создаем копию для обработки
test_data_processed = test_data.copy()

# Определяем численные и категориальные признаки (как в задании 5)
numerical_cols_test = test_data_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_test = test_data_processed.select_dtypes(include=['object']).columns.tolist()

# Удаляем Id из численных
if 'Id' in numerical_cols_test:
    numerical_cols_test.remove('Id')

print(f"\nЧисленные признаки: {len(numerical_cols_test)}")
print(f"Категориальные признаки: {len(categorical_cols_test)}")

# Обработка пропущенных значений (та же логика, что в задании 5)
print("\nОбработка пропущенных значений...")

# Список признаков, где "NA" означает "None"
na_means_none = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature'
]

# Обработка численных признаков
for col in numerical_cols_test:
    if col in test_data_processed.columns and test_data_processed[col].isnull().sum() > 0:
        # Используем медиану из обучающих данных, если доступна, иначе из тестовых
        if col in house_data_processed.columns:
            median_value = house_data_processed[col].median()
        else:
            median_value = test_data_processed[col].median()
        test_data_processed[col].fillna(median_value, inplace=True)

# Обработка категориальных признаков
for col in categorical_cols_test:
    if col in test_data_processed.columns:
        nan_count = test_data_processed[col].isnull().sum()
        if nan_count > 0:
            if col in na_means_none:
                test_data_processed[col].fillna('None', inplace=True)
            else:
                # Используем моду из обучающих данных, если доступна
                if col in house_data_processed.columns:
                    mode_value = house_data_processed[col].mode()
                    if len(mode_value) > 0:
                        test_data_processed[col].fillna(mode_value[0], inplace=True)
                    else:
                        test_data_processed[col].fillna('Unknown', inplace=True)
                else:
                    mode_value = test_data_processed[col].mode()
                    if len(mode_value) > 0:
                        test_data_processed[col].fillna(mode_value[0], inplace=True)
                    else:
                        test_data_processed[col].fillna('Unknown', inplace=True)

print("✓ Пропущенные значения обработаны")

# Преобразование категориальных данных (та же логика, что в задании 5)
print("\nПреобразование категориальных данных...")

# Определяем порядковые признаки (те же, что в задании 5)
ordinal_features = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'Utilities': ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    'LandContour': ['Low', 'Bnk', 'HLS', 'Lvl']
}

# Применяем ординальное кодирование
for col, categories in ordinal_features.items():
    if col in test_data_processed.columns:
        category_map = {cat: idx for idx, cat in enumerate(categories)}
        test_data_processed[col] = test_data_processed[col].map(category_map)
        if test_data_processed[col].isnull().sum() > 0:
            # Заполняем медианой из обучающих данных или тестовых
            if col in house_data_processed.columns:
                median_val = house_data_processed[col].median()
            else:
                median_val = test_data_processed[col].median()
            test_data_processed[col].fillna(median_val, inplace=True)
        test_data_processed[col] = test_data_processed[col].astype(int)

# One-Hot Encoding для остальных категориальных признаков
categorical_cols_remaining = [col for col in categorical_cols_test 
                             if col in test_data_processed.columns 
                             and col not in ordinal_features.keys()]

if len(categorical_cols_remaining) > 0:
    # Используем те же категории, что были в обучающих данных
    encoded_cols = pd.get_dummies(
        test_data_processed[categorical_cols_remaining], 
        prefix=categorical_cols_remaining,
        drop_first=True,
        dummy_na=False
    )
    
    # Удаляем исходные категориальные столбцы
    test_data_processed = test_data_processed.drop(columns=categorical_cols_remaining)
    
    # Добавляем закодированные столбцы
    test_data_processed = pd.concat([test_data_processed, encoded_cols], axis=1)

# Удаляем Id
if 'Id' in test_data_processed.columns:
    test_data_processed = test_data_processed.drop(columns=['Id'])

print(f"✓ Категориальные признаки преобразованы")
print(f"Размер после предобработки: {test_data_processed.shape}")

# Выбираем только те признаки, которые есть в selected_features
print("\nВыбор признаков из selected_features...")
missing_features = [f for f in selected_features if f not in test_data_processed.columns]
if len(missing_features) > 0:
    print(f"⚠️ Отсутствуют признаки в тестовых данных ({len(missing_features)}): {missing_features[:5]}...")
    # Заполняем отсутствующие признаки нулями
    for feat in missing_features:
        test_data_processed[feat] = 0

# Убираем признаки, которых нет в selected_features
extra_features = [f for f in test_data_processed.columns if f not in selected_features]
if len(extra_features) > 0:
    test_data_processed = test_data_processed.drop(columns=extra_features)

# Убеждаемся, что порядок признаков совпадает с X_selected
test_data_processed = test_data_processed[selected_features]

print(f"✓ Отобрано признаков: {test_data_processed.shape[1]}")
print(f"✓ Размерность совпадает с X_selected: {test_data_processed.shape[1] == X_selected.shape[1]}")

# 2. Загрузка истинных значений для теста
print("\n" + "=" * 60)
print("2. ЗАГРУЗКА ИСТИННЫХ ЗНАЧЕНИЙ ДЛЯ ТЕСТА")
print("=" * 60)

sample_submission_path = 'house-prices-advanced/sample_submission.csv'
sample_submission = pd.read_csv(sample_submission_path)

print(f"Загружено записей из sample_submission: {len(sample_submission)}")
print(f"Колонки: {sample_submission.columns.tolist()}")

# Извлекаем SalePrice для тестовых данных
y_test_true = sample_submission['SalePrice'].values

print(f"Количество истинных значений: {len(y_test_true)}")
print(f"Диапазон значений: от {y_test_true.min():,.0f} до {y_test_true.max():,.0f}")

# 3. Вычисление RMSE
print("\n" + "=" * 60)
print("3. ВЫЧИСЛЕНИЕ RMSE")
print("=" * 60)

# Предсказания на тренировочных данных (используем все X_selected, так как модель обучена на части)
y_train_pred = elastic_net_model.predict(X_selected)
y_train_true = y_house.values

# Предсказания на тестовых данных
y_test_pred = elastic_net_model.predict(test_data_processed)

# Вычисление RMSE
rmse_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test_true, y_test_pred))

# Общая ошибка (RMSE на объединенных данных)
y_all_true = np.concatenate([y_train_true, y_test_true])
y_all_pred = np.concatenate([y_train_pred, y_test_pred])
rmse_overall = np.sqrt(mean_squared_error(y_all_true, y_all_pred))

print(f"\nRMSE на тренировочных данных: {rmse_train:.2f}")
print(f"RMSE на тестовой выборке: {rmse_test:.2f}")
print(f"Общая ошибка (RMSE): {rmse_overall:.2f}")

# 4. Визуализация результатов
print("\n" + "=" * 60)
print("4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("=" * 60)

# График 1: Сравнение предсказанных и реальных значений для train
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(y_train_true, y_train_pred, alpha=0.5, s=20)
plt.plot([y_train_true.min(), y_train_true.max()], 
         [y_train_true.min(), y_train_true.max()], 'r--', lw=2, label='Идеальная линия')
plt.xlabel('Реальные значения SalePrice', fontsize=10)
plt.ylabel('Предсказанные значения SalePrice', fontsize=10)
plt.title(f'Тренировочные данные\nRMSE = {rmse_train:.2f}', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# График 2: Сравнение предсказанных и реальных значений для test
plt.subplot(1, 3, 2)
plt.scatter(y_test_true, y_test_pred, alpha=0.5, s=20, color='green')
plt.plot([y_test_true.min(), y_test_true.max()], 
         [y_test_true.min(), y_test_true.max()], 'r--', lw=2, label='Идеальная линия')
plt.xlabel('Реальные значения SalePrice', fontsize=10)
plt.ylabel('Предсказанные значения SalePrice', fontsize=10)
plt.title(f'Тестовая выборка\nRMSE = {rmse_test:.2f}', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# График 3: Распределение ошибок (residuals)
plt.subplot(1, 3, 3)
residuals_train = y_train_true - y_train_pred
residuals_test = y_test_true - y_test_pred
plt.hist(residuals_train, bins=50, alpha=0.6, label='Train', color='blue', edgecolor='black')
plt.hist(residuals_test, bins=50, alpha=0.6, label='Test', color='green', edgecolor='black')
plt.xlabel('Ошибка (Residuals)', fontsize=10)
plt.ylabel('Частота', fontsize=10)
plt.title('Распределение ошибок', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# График 4: Сравнение RMSE train vs test
plt.figure(figsize=(10, 6))

plt.subplot(2, 2, 1)
rmse_values = [rmse_train, rmse_test, rmse_overall]
rmse_labels = ['Train', 'Test', 'Overall']
colors_bar = ['blue', 'green', 'red']
plt.bar(rmse_labels, rmse_values, color=colors_bar, alpha=0.7, edgecolor='black')
plt.ylabel('RMSE', fontsize=12)
plt.title('Сравнение RMSE', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')
for i, (label, value) in enumerate(zip(rmse_labels, rmse_values)):
    plt.text(i, value + max(rmse_values)*0.02, f'{value:.2f}', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# График 5: График остатков (residuals vs predicted) для train
plt.subplot(2, 2, 2)
plt.scatter(y_train_pred, residuals_train, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Предсказанные значения', fontsize=10)
plt.ylabel('Ошибки (Residuals)', fontsize=10)
plt.title('Остатки: Тренировочные данные', fontsize=12)
plt.grid(True, alpha=0.3)

# График 6: График остатков (residuals vs predicted) для test
plt.subplot(2, 2, 3)
plt.scatter(y_test_pred, residuals_test, alpha=0.5, s=20, color='green')
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Предсказанные значения', fontsize=10)
plt.ylabel('Ошибки (Residuals)', fontsize=10)
plt.title('Остатки: Тестовая выборка', fontsize=12)
plt.grid(True, alpha=0.3)

# График 7: Сравнение распределений предсказаний и реальных значений
plt.subplot(2, 2, 4)
plt.hist(y_train_true, bins=50, alpha=0.6, label='Train (реальные)', color='blue', edgecolor='black')
plt.hist(y_train_pred, bins=50, alpha=0.6, label='Train (предсказанные)', color='orange', edgecolor='black')
plt.xlabel('SalePrice', fontsize=10)
plt.ylabel('Частота', fontsize=10)
plt.title('Распределение значений: Train', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Дополнительная статистика
print("\n" + "=" * 60)
print("СТАТИСТИКА ПО ОШИБКАМ")
print("=" * 60)

print(f"\nТренировочные данные:")
print(f"  Средняя ошибка: {np.mean(residuals_train):.2f}")
print(f"  Медианная ошибка: {np.median(residuals_train):.2f}")
print(f"  Стандартное отклонение ошибок: {np.std(residuals_train):.2f}")

print(f"\nТестовая выборка:")
print(f"  Средняя ошибка: {np.mean(residuals_test):.2f}")
print(f"  Медианная ошибка: {np.median(residuals_test):.2f}")
print(f"  Стандартное отклонение ошибок: {np.std(residuals_test):.2f}")

print("\n" + "=" * 60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("=" * 60)
print(f"""
RMSE на тренировочных данных: {rmse_train:.2f}
RMSE на тестовой выборке: {rmse_test:.2f}
Общая ошибка (RMSE): {rmse_overall:.2f}

Разница между train и test RMSE: {abs(rmse_train - rmse_test):.2f}
""")


### Выводы

1. **Предобработка тестовых данных**: 
   - Применена та же логика предобработки, что и для обучающих данных
   - Обработаны пропущенные значения (медиана для численных, мода/None для категориальных)
   - Применено ординальное и One-Hot кодирование категориальных признаков
   - Отобраны только признаки из selected_features

2. **Загрузка истинных значений**: 
   - Загружены истинные значения SalePrice из sample_submission.csv
   - Значения сопоставлены с тестовыми данными

3. **Вычисление RMSE**: 
   - **RMSE на тренировочных данных**: оценка качества модели на обучающих данных
   - **RMSE на тестовой выборке**: оценка обобщающей способности модели
   - **Общая ошибка**: RMSE на объединенных данных (train + test)

4. **Визуализация результатов**: 
   - Построены scatter plots для сравнения предсказанных и реальных значений
   - Визуализировано распределение ошибок (residuals)
   - Построены графики остатков для диагностики модели
   - Сравнены RMSE для разных выборок

5. **Результат**: Получена полная оценка качества модели Elastic Net с визуализацией результатов
